# Historical RAG Pipeline Experiment

> Original end-to-end experiment retained for reproducibility; superseded by the modular pipeline in `src/rag.py`.

## 1. Install Dependencies

In [1]:
!pip install -q unsloth
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes

!pip install -q \
langchain==0.3.25 \
langchain-community==0.3.24 \
langchain-core==0.3.86 \
langchain-text-splitters==0.3.8 \
langchain-huggingface==0.1.2

!pip install -q \
faiss-cpu==1.9.0 \
rank-bm25==0.2.2 \
sentence-transformers==3.4.1

!pip install -q \
pypdf \
pymupdf==1.25.5

!pip install -q duckduckgo-search==8.0.1

!pip install -q \
gdown \
gradio==5.30.0

!pip install -q \
pandas \
numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.5/27.5 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 63.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curre

# 2. Import Libraries

In [2]:
import os
import torch
from transformers import pipeline

from unsloth import FastLanguageModel
from sentence_transformers import CrossEncoder

from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryByteStore
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever

from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain.retrievers import ContextualCompressionRetriever

from langchain_community.tools import DuckDuckGoSearchRun
from duckduckgo_search import DDGS

from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU: Tesla T4


# 3. Load Model

In [3]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Slotherynn/legal-chatbot-qwen-grpo",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    dtype=None,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048, padding_idx=151665)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen

In [4]:
folder_id = "1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql"
os.system(f"gdown --folder https://drive.google.com/drive/folders/{folder_id} -O /content/legal_docs")

PDF_FILES = [
    "/content/legal_docs/PP Nomor 5 Tahun 2021.pdf",
    "/content/legal_docs/PP Nomor 35 Tahun 2021.pdf",
    "/content/legal_docs/PP Nomor 51 Tahun 2023.pdf",
    "/content/legal_docs/UU Nomor 6 Tahun 2023.pdf",
]

try:
    from langchain_community.document_loaders import PyPDFLoader
except ModuleNotFoundError:
    from langchain.document_loaders import PyPDFLoader

all_documents = []
for path in PDF_FILES:
    loader = PyPDFLoader(path)
    docs = loader.load()
    all_documents.extend(docs)
    print(f"{os.path.basename(path)}: {len(docs)} halaman")

print(f"\nTotal halaman: {len(all_documents)}")

PP Nomor 5 Tahun 2021.pdf: 739 halaman
PP Nomor 35 Tahun 2021.pdf: 56 halaman
PP Nomor 51 Tahun 2023.pdf: 27 halaman
UU Nomor 6 Tahun 2023.pdf: 1127 halaman

Total halaman: 1949


# 4. Metadata Enrichment

In [5]:
DOC_META = {
    "PP Nomor 5 Tahun 2021.pdf":  {"peraturan": "PP No.5/2021",  "tahun": 2021, "topik": "Perizinan Berusaha"},
    "PP Nomor 35 Tahun 2021.pdf": {"peraturan": "PP No.35/2021", "tahun": 2021, "topik": "Ketenagakerjaan PKWT & Alih Daya"},
    "PP Nomor 51 Tahun 2023.pdf": {"peraturan": "PP No.51/2023", "tahun": 2023, "topik": "Pengupahan"},
    "UU Nomor 6 Tahun 2023.pdf":  {"peraturan": "UU No.6/2023",  "tahun": 2023, "topik": "Cipta Kerja"},
}

for doc in all_documents:
    fname = os.path.basename(doc.metadata.get("source", ""))
    extra = DOC_META.get(fname, {})
    doc.metadata.update(extra)

print("Metadata enrichment selesai!")
print(f"Contoh metadata: {all_documents[0].metadata}")

Metadata enrichment selesai!
Contoh metadata: {'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2021-04-22T13:25:58+07:00', 'author': 'Lenovo75', 'moddate': '2021-04-22T18:03:50+07:00', 'source': '/content/legal_docs/PP Nomor 5 Tahun 2021.pdf', 'total_pages': 739, 'page': 0, 'page_label': '1', 'peraturan': 'PP No.5/2021', 'tahun': 2021, 'topik': 'Perizinan Berusaha'}


# 5. Strukturisasi Parent-Child Retriever & Penyimpanan Vektor Lokal

In [6]:
# Parent: chunk besar untuk konteks LLM
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
)

# Child: chunk kecil untuk pencarian vektor
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

# Embedding model open-source
print("Memuat embedding model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding model ready.")

# FAISS vectorstore untuk child chunks
vectorstore = FAISS.from_documents(
    child_splitter.split_documents(all_documents),
    embedding_model
)
print("FAISS vectorstore ready.")

# InMemoryByteStore untuk parent chunks
docstore = InMemoryByteStore()

# ParentDocumentRetriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_type="similarity",
    search_kwargs={"k": 10}
)
parent_retriever.add_documents(all_documents)
print("ParentDocumentRetriever ready.")

Memuat embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Embedding model ready.
FAISS vectorstore ready.
ParentDocumentRetriever ready.


# 6. Ensemble Retriever

In [7]:
parent_docs_for_bm25 = parent_splitter.split_documents(all_documents)

bm25_retriever = BM25Retriever.from_documents(parent_docs_for_bm25)
bm25_retriever.k = 10

print("BM25 Retriever siap.")

# Ensemble: gabungan BM25 (keyword) + ParentDocumentRetriever (semantic)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, parent_retriever],
    weights=[0.4, 0.6]
)

print("Ensemble Retriever (Hybrid) siap.")
print(f"Bobot: BM25=0.4 | ParentDocumentRetriever=0.6")

BM25 Retriever siap.
Ensemble Retriever (Hybrid) siap.
Bobot: BM25=0.4 | ParentDocumentRetriever=0.6


# 7. Load Reranker Model

In [8]:
reranker_model = CrossEncoder('BAAI/bge-reranker-base')

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

# 8. HyDe Berbasis Qwen

In [9]:
def generate_hyde_hypothetical_answers(query):
    hyde_prompt = f"""<|im_start|>user
Berikan jawaban hipotetis singkat dalam bahasa Indonesia yang mungkin berisi pasal-pasal hukum terkait pertanyaan berikut: {query}<|im_end|>
<|im_start|>assistant
"""
    inputs = tokenizer(
        [hyde_prompt, hyde_prompt],
        return_tensors="pt",
        padding=True
    ).to("cuda")

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    hypothetical_docs = [
        tokenizer.decode(output[input_len:], skip_special_tokens=True)
        for output in outputs
    ]

    print(f"Jawaban hipotetis 1: {hypothetical_docs[0][:100]}...")
    print(f"Jawaban hipotetis 2: {hypothetical_docs[1][:100]}...")

    return hypothetical_docs

# 9. Advanced Retrieval Pipeline (Reranker + DuckDuckGo Fallback)

In [10]:
THRESHOLD = 0.1

def duckduckgo_fallback(query):
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(
                query + " hukum ketenagakerjaan Indonesia",
                max_results=3
            ))
        if results:
            combined = "\n\n".join([
                f"{r.get('title','')}: {r.get('body','')}"
                for r in results
            ])
            return combined
        return "Tidak ada hasil dari internet."
    except Exception as e:
        return f"DuckDuckGo gagal: {str(e)}"

def advanced_retrieval_pipeline(query):
    # Step 1: HyDE
    hypo_answers = generate_hyde_hypothetical_answers(query)
    combined_search_query = f"{query} " + " ".join(hypo_answers)

    # Step 2: Hybrid retrieve
    retrieved_docs = hybrid_retriever.invoke(combined_search_query)[:6]

    if not retrieved_docs:
        print("Tidak ada dokumen → fallback internet")
        return [duckduckgo_fallback(query)], "Fallback DuckDuckGo (No Docs Found)"

    # Step 3: Reranking
    pairs = [[query, doc.page_content] for doc in retrieved_docs]
    scores = reranker_model.predict(pairs)
    scored_docs = sorted(zip(scores, retrieved_docs), key=lambda x: x[0], reverse=True)

    # Step 4: Ekstrak Top-1 score → gate
    top_1_score = float(scored_docs[0][0])
    print(f"Reranker Top-1 Score: {top_1_score:.4f} (threshold={THRESHOLD})")

    if top_1_score < THRESHOLD:
        print("Relevansi rendah → beralih ke DuckDuckGo...")
        return [duckduckgo_fallback(query)], "Sumber Web Eksternal (Internet Fallback)"
    else:
        final_chunks = [doc for _, doc in scored_docs[:3]]
        return final_chunks, "Dokumen Hukum Internal"

print("Advanced Retrieval Pipeline siap.")

# Quick test
docs, source = advanced_retrieval_pipeline("upah lembur pekerja harian")
print(f"   Sumber: {source} | Dokumen: {len(docs)}")

Advanced Retrieval Pipeline siap.
Jawaban hipotetis 1: Jika seorang pekerja harian bekerja lembur, upah lembur tersebut harus dihitung dan dibayarkan sesua...
Jawaban hipotetis 2: Pertanyaan ini berkaitan dengan upah lembur untuk pekerja harian....
Reranker Top-1 Score: 0.9527 (threshold=0.1)
   Sumber: Dokumen Hukum Internal | Dokumen: 3


# 10. Interface & Study Case

In [11]:
from IPython.display import display, Markdown

def run_chatbot_rag_app(user_prompt):
    # Step 1: Retrieval
    context_chunks, source_type = advanced_retrieval_pipeline(user_prompt)

    # Step 2: Format context + sitasi
    if isinstance(context_chunks[0], str):
        # Hasil dari DuckDuckGo fallback (string bukan Document)
        context_text = context_chunks[0]
        citations = "Konteks bersumber dari pencarian web eksternal (DuckDuckGo)."
    else:
        context_text = "\n\n".join([c.page_content for c in context_chunks])
        citations = ", ".join(list(set([
            f"{c.metadata.get('source_file', '?')} (Hal. {c.metadata.get('page', '?')})"
            for c in context_chunks
        ])))

    # Step 3: Prompt dengan chain-of-thought <think> tag
    final_rag_prompt = f"""<|im_start|>system
Kamu adalah Legal Compliance Assistant yang membantu tim legal perusahaan dalam memahami regulasi Indonesia.

Sebelum menjawab:
1. Analisis dokumen.
2. Tampilkan reasoning pada tag <think>.
3. Cantumkan kesimpulan yang didukung regulasi.
4. Jangan mengarang informasi yang tidak ditemukan pada konteks.
<|im_start|>user
Gunakan Konteks Regulasi Berikut untuk Menjawab:
{context_text}

Pertanyaan Pengguna: {user_prompt}<|im_end|>
<|im_start|>assistant
"""

    # Step 4: Generate
    inputs = tokenizer([final_rag_prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.2,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode hanya bagian baru (bukan prompt)
    response_text = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[-1]:],
        skip_special_tokens=True
    )

    # Step 5: Tampilkan output
    display(Markdown(f"### Respons Asisten AI (Model: Qwen2.5-GRPO-Legal) [{source_type}]"))
    display(Markdown(response_text))
    display(Markdown(f"\n**Sitasi Referensi Dokumen Terkait:** *{citations}*"))


# --- TEST CASE WAJIB ---
test_prompt = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"
run_chatbot_rag_app(test_prompt)

Jawaban hipotetis 1: Ya, Anda berhak mendapatkan uang lembur sesuai dengan ketentuan perusahaan dan aturan pajak yang ber...
Jawaban hipotetis 2: Ya, Anda berhak mendapatkan gaji lembur karena Anda telah bekerja lebih dari jam kerja standar. Namu...
Reranker Top-1 Score: 0.4815 (threshold=0.1)


### Respons Asisten AI (Model: Qwen2.5-GRPO-Legal) [Dokumen Hukum Internal]

Ya, Anda berhak mendapatkan upah lembur. Sebagai staf admin, Anda memiliki hak atas upah lembur sesuai dengan aturan yang ditetapkan oleh perusahaan Anda.


**Sitasi Referensi Dokumen Terkait:** *? (Hal. 558), ? (Hal. 18), ? (Hal. 16)*